In [0]:
spark.sql("""
SELECT *,
  upper(customer_name) as Customer_Name_Upper,
  date(current_timestamp()) as processDate
FROM datamodeling.bronze.bronze_table""").createOrReplaceTempView("silver_source")

In [0]:
%sql
SELECT * FROM silver_source

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_update,Customer_Name_Upper,processDate
1004,2024-07-02,4,David Lee,david@abc.com,504,Samsung S23,Electronics,1,899.99,Credit Card,USA,2024-07-02,DAVID LEE,2025-11-05
1005,2024-07-02,1,Alice Johnson,alice@gmail.com,503,Nike Shoes,Footwear,2,129.99,Credit Card,USA,2024-07-02,ALICE JOHNSON,2025-11-05


# MERGE Using Pyspark

In [0]:
if spark.catalog.tableExists('datamodeling.silver.silver_table'):
    pass

else:
    spark.sql("""
              CREATE TABLE IF NOT EXISTS datamodeling.silver.silver_table
              AS
              SELECT * FROM silver_source
              """)

# MERGE Using SQL


In [0]:
%sql
MERGE INTO datamodeling.silver.silver_table
USING silver_source
ON datamodeling.silver.silver_table.order_id = silver_source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT*

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
%sql
SELECT * FROM datamodeling.silver.silver_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_update,Customer_Name_Upper,processDate
1004,2024-07-02,4,David Lee,david@abc.com,504,Samsung S23,Electronics,1,899.99,Credit Card,USA,2024-07-02,DAVID LEE,2025-11-05
1005,2024-07-02,1,Alice Johnson,alice@gmail.com,503,Nike Shoes,Footwear,2,129.99,Credit Card,USA,2024-07-02,ALICE JOHNSON,2025-11-05
1001,2024-07-01,1,Alice Johnson,alice@gmail.com,501,iPhone 14,Electronics,1,999.99,Credit Card,USA,2024-07-01,ALICE JOHNSON,2025-11-05
1002,2024-07-01,2,Bob Smith,bob@yahoo.com,502,AirPods Pro,Electronics,2,199.99,PayPal,USA,2024-07-01,BOB SMITH,2025-11-05
1003,2024-07-01,3,Charlie Brown,charlie@outlook.com,503,Nike Shoes,Footwear,1,129.99,Credit Card,Canada,2024-07-01,CHARLIE BROWN,2025-11-05
